# Fine-tune Wav2Vec2 (CTC) on a South African language — local RTX 3060

**Anti-collapse + speed pass.** Changes from the previous version, and why:

1. **Length-feasibility filter turned ON** (was commented out). Samples where
   the downsampled audio has fewer frames than the label needs are CTC-infeasible
   and push the model toward degenerate "predict blank everywhere" shortcuts.
2. **Phased encoder freezing.** Only the top `N` transformer layers (+ `lm_head`)
   train initially; the rest of the 24-layer encoder stays frozen. ~300M free
   parameters against ~200 examples was enough to collapse in under 40 steps —
   cutting trainable capacity buys you a more stable starting point. You can
   unfreeze more layers once training is stable and you've scaled up the dataset.
3. **Real warmup.** `warmup_ratio=0.1` over 39 total steps was ~4 steps — LR
   was essentially at full value immediately. Now sized relative to actual
   total steps.
4. **bf16 instead of fp16.** Ampere (RTX 3060) supports bf16 natively; it has
   fp32's exponent range so it doesn't need loss scaling and is less prone to
   the instability that fp16 can produce on CTC's often-huge unnormalized loss values.
5. **Speed**: `group_by_length=True` to cut padding waste (audio clips vary a lot
   in length — padding to the longest in a random batch wastes real compute),
   `optim="adamw_torch_fused"` for a faster fused CUDA optimizer step, and
   `dataloader_num_workers` set to use your CPU cores instead of blocking on I/O.

**Expected data layout**: same as before — HF hub dataset
`dsfsi-anv/za-african-next-voices-compressed`, config = target language,
`train` / `dev_test` / `dev` splits, `transcript` field.

## 1. Install dependencies

In [1]:
import multiprocessing as mp
mp.set_start_method("fork", force=True)

In [2]:
!pip install -q transformers datasets evaluate jiwer accelerate soundfile librosa torchcodec

## 2. Check GPU

In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM (GB): 12.5
bf16 supported: True


## 3. Config — edit these

In [4]:
LANGUAGE = "tsn"  # tsn=Setswana, nso=Sepedi, ven=Tshivenda
BASE_MODEL = "facebook/wav2vec2-xls-r-300m"
OUTPUT_DIR = f"./wav2vec2-300-{LANGUAGE}"



# Phased freezing: only the top N transformer encoder layers are trainable
# (out of 24 in wav2vec2-xls-r-300m), plus lm_head. Raise this once a
# small run looks stable and you've scaled the dataset up.
NUM_TRAINABLE_ENCODER_LAYERS = 4

CHARS_TO_IGNORE = r'[,\?\.!\-\;\:"“%‘”�0-9\[\]\'\_]'

## 4. Imports

In [5]:
import re
import json
import numpy as np
import torch
from dataclasses import dataclass
from typing import Dict, List, Union
from datasets import load_dataset, Audio, Dataset
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)
import evaluate

/home/khotso/projects/MultilingualASR/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5. Load and normalize data

In [6]:
dataset_dict = load_dataset(
    "dsfsi-anv/za-african-next-voices-compressed",
    LANGUAGE,
)

In [7]:
def filter_by_duration(dataset_dict, min_seconds=5.0, max_seconds=10.0, splits=None, num_proc=1):
    """
    Filters the given DatasetDict in-place keeping examples with duration in [min_seconds, max_seconds].
    - dataset_dict: datasets.DatasetDict
    - splits: list of split names to filter (None => all splits)
    - num_proc: number of processes for `.filter()`
    """
    if splits is None:
        splits = list(dataset_dict.keys())

    def _in_range(example):
        d = example.get("duration")
        return d is not None and (min_seconds <= d <= max_seconds)

    for split in splits:
        dataset_dict[split] = dataset_dict[split].filter(_in_range, num_proc=num_proc)

    return dataset_dict

In [8]:
from collections import defaultdict
import math

def duration_buckets(durations, bucket_size=1):
    """
    Bucket audio durations into fixed-width bins.

    Args:
        durations (list/iterable of float): duration in seconds per sample.
        bucket_size (float): width of each bucket in seconds.

    Returns:
        dict: {bucket_start: count}, sorted by bucket_start.
    """
    buckets = defaultdict(int)
    for d in durations:
        bucket_start = math.floor(d / bucket_size) * bucket_size
        buckets[bucket_start] += 1
    return dict(sorted(buckets.items()))


def print_duration_buckets(dataset_dict, splits=("train", "dev"), bucket_size=1):
    for split in splits:
        print(split)
        durations = dataset_dict[split]["duration"]
        buckets = duration_buckets(durations, bucket_size)

        total_dur = sum(durations)
        print(f"total duration: {total_dur} seconds")
        print(f"total duration: {total_dur/60} minutes")
        print(f"total duration: {total_dur/3600} hours")
        print(f"total samples: {len(durations)}")
        print()

        for bucket_start, count in buckets.items():
            bucket_end = bucket_start + bucket_size
            pct = 100 * count / len(durations)
            print(f"  {bucket_start:>5.1f} - {bucket_end:>5.1f} sec: {count:>6} samples ({pct:5.1f}%)")
        print()

In [9]:
print_duration_buckets(dataset_dict, splits=["train", "dev","dev_test"], bucket_size=1)

train
total duration: 1532637.609391 seconds
total duration: 25543.960156516663 minutes
total duration: 425.7326692752778 hours
total samples: 84383

    2.0 -   3.0 sec:    139 samples (  0.2%)
    3.0 -   4.0 sec:   1611 samples (  1.9%)
    4.0 -   5.0 sec:   5313 samples (  6.3%)
    5.0 -   6.0 sec:   8033 samples (  9.5%)
    6.0 -   7.0 sec:   7318 samples (  8.7%)
    7.0 -   8.0 sec:   5552 samples (  6.6%)
    8.0 -   9.0 sec:   4034 samples (  4.8%)
    9.0 -  10.0 sec:   3230 samples (  3.8%)
   10.0 -  11.0 sec:   2678 samples (  3.2%)
   11.0 -  12.0 sec:   2500 samples (  3.0%)
   12.0 -  13.0 sec:   2296 samples (  2.7%)
   13.0 -  14.0 sec:   2205 samples (  2.6%)
   14.0 -  15.0 sec:   2173 samples (  2.6%)
   15.0 -  16.0 sec:   1967 samples (  2.3%)
   16.0 -  17.0 sec:   1797 samples (  2.1%)
   17.0 -  18.0 sec:   1796 samples (  2.1%)
   18.0 -  19.0 sec:   1720 samples (  2.0%)
   19.0 -  20.0 sec:   1618 samples (  1.9%)
   20.0 -  21.0 sec:   1693 samples (  2

In [10]:
dataset_dict = filter_by_duration(dataset_dict, min_seconds=4, max_seconds=15, splits=["train","dev"], num_proc=2)

In [11]:
print_duration_buckets(dataset_dict, splits=["train", "dev"], bucket_size=1)


train


total duration: 368935.750349 seconds
total duration: 6148.929172483333 minutes
total duration: 102.48215287472222 hours
total samples: 45332

    4.0 -   5.0 sec:   5313 samples ( 11.7%)
    5.0 -   6.0 sec:   8033 samples ( 17.7%)
    6.0 -   7.0 sec:   7318 samples ( 16.1%)
    7.0 -   8.0 sec:   5552 samples ( 12.2%)
    8.0 -   9.0 sec:   4034 samples (  8.9%)
    9.0 -  10.0 sec:   3230 samples (  7.1%)
   10.0 -  11.0 sec:   2678 samples (  5.9%)
   11.0 -  12.0 sec:   2500 samples (  5.5%)
   12.0 -  13.0 sec:   2296 samples (  5.1%)
   13.0 -  14.0 sec:   2205 samples (  4.9%)
   14.0 -  15.0 sec:   2173 samples (  4.8%)

dev
total duration: 20358.591881 seconds
total duration: 339.30986468333333 minutes
total duration: 5.655164411388889 hours
total samples: 2778

    4.0 -   5.0 sec:    472 samples ( 17.0%)
    5.0 -   6.0 sec:    632 samples ( 22.8%)
    6.0 -   7.0 sec:    528 samples ( 19.0%)
    7.0 -   8.0 sec:    345 samples ( 12.4%)
    8.0 -   9.0 sec:    191 samples 

In [12]:
def sample_n(dataset_dict, target_totals, seed=42):
    for split, n in target_totals.items():
        ds = dataset_dict[split]
        n = min(n, len(ds))
        dataset_dict[split] = ds.shuffle(seed=seed).select(range(n))
    return dataset_dict



In [13]:
dataset_dict = sample_n(dataset_dict, {"train": 4500, "dev": 500})

In [14]:
for split in ["train", "dev","dev_test"]:
    dataset_dict[split] = dataset_dict[split].cast_column("audio", Audio(sampling_rate=16000))

#dataset_dict= dataset_dict.cast_column("audio", Audio(sampling_rate=16000))

In [15]:
print_duration_buckets(dataset_dict, splits=["train", "dev"], bucket_size=1)

train
total duration: 36675.335538 seconds
total duration: 611.2555923 minutes
total duration: 10.187593205 hours
total samples: 4500

    4.0 -   5.0 sec:    555 samples ( 12.3%)
    5.0 -   6.0 sec:    774 samples ( 17.2%)
    6.0 -   7.0 sec:    707 samples ( 15.7%)
    7.0 -   8.0 sec:    575 samples ( 12.8%)
    8.0 -   9.0 sec:    385 samples (  8.6%)
    9.0 -  10.0 sec:    337 samples (  7.5%)
   10.0 -  11.0 sec:    241 samples (  5.4%)
   11.0 -  12.0 sec:    264 samples (  5.9%)
   12.0 -  13.0 sec:    202 samples (  4.5%)
   13.0 -  14.0 sec:    230 samples (  5.1%)
   14.0 -  15.0 sec:    230 samples (  5.1%)

dev
total duration: 3639.090397 seconds
total duration: 60.651506616666666 minutes
total duration: 1.010858443611111 hours
total samples: 500

    4.0 -   5.0 sec:     93 samples ( 18.6%)
    5.0 -   6.0 sec:    118 samples ( 23.6%)
    6.0 -   7.0 sec:     91 samples ( 18.2%)
    7.0 -   8.0 sec:     63 samples ( 12.6%)
    8.0 -   9.0 sec:     31 samples (  6.2%)
 

In [16]:
import random



In [17]:
def normalize_text(batch):
    if batch["transcript"] is None:
        return batch

    text = batch["transcript"]

    # Remove annotation tags like [pause], [cs], [?], [noise] etc.
    text = re.sub(r'\[.*?\]', '', text)

    # Lowercase everything
    text = text.lower()

    # Keep: a-z, apostrophe, whitespace, and the specific diacritic
    # characters confirmed to exist in this corpus:
    # ê ñ ô ŝ š ȇ ȏ ḓ
    # Everything else (digits, punctuation like ! " ? _ , - etc.)
    # becomes a space rather than being deleted, to avoid gluing words together
    text = re.sub(r"[^a-z'êñôŝšȇȏḓ\s]", " ", text)

    # Collapse repeated whitespace and trim ends
    text = re.sub(r'\s+', ' ', text).strip()

    batch["transcript"] = text
    return batch

In [18]:
def is_valid_transcript(batch):
    t = batch["transcript"]
    return t is not None and isinstance(t, str) and t.strip() != ""

In [19]:
print(len(dataset_dict['train']))

4500


In [20]:
for split in ["train", "dev", "dev_test"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=4)

In [21]:
print(len(dataset_dict['train']))

4497


In [22]:
random.seed(42)
random_indices = random.sample(range(len(dataset_dict["train"])), 10)
print("Random indices:", random_indices)
for i, idx in enumerate(random_indices):
    print(f'{i+1}: {dataset_dict["train"][idx]["transcript"]}')

Random indices: [912, 204, 2253, 2006, 1828, 1143, 839, 4467, 712, 3456]
1: Mdletshe o rata go nna mo metsing. Metsi a kgona go go tlhalosetsa go le gontsi ka seo maemo a letsatsi a se go tsholetseng.
2: O tlhalositse gore dikolo tse di sa akarediwang mo lenaaneng leno di ikopanye le dikantoro tsa kgaolo fa e le gore le tsona di batla go akarediwa.
3: Basimane ba rata go tshameka kgwele ya dinaȏ kwa ntle.
4: Go tshwanetse go nne le lenaneo le le tla rutang bagolo, ka dilo tsa go tshwana le tse.
5: Basimane ba rata go tshameka kgwele ya dinaȏ kwa ntle.
6: Di diriwa ka letlalo la kgomo kgotsa letlalo la podi kgotsa la nku.
7: Ke nagana gore motho o tshwanetse [?] go rulaganya, go [?].
8: Seno se re ama rotlhe, mme re tshwanetse go jara maikarabelo mmogo mme rotlhe re thusane go goga le go ntsha naga ya rona mo tobetobeng e e leng mo go yona.
9: Botshelo jwa diritibatsi ke bo tlisitse mathata mo basimaneng ba bantsi.
10: Basimane ba tshameka motshameko wa modikologȏ mo lebaleng.


In [23]:
for split in ["train", "dev","dev_test"]:
    dataset_dict[split] = dataset_dict[split].map(normalize_text)

In [24]:
for i,idx in enumerate(random_indices):
    print(f'{i+1}: {dataset_dict["train"][idx]["transcript"]}')

1: mdletshe o rata go nna mo metsing metsi a kgona go go tlhalosetsa go le gontsi ka seo maemo a letsatsi a se go tsholetseng
2: o tlhalositse gore dikolo tse di sa akarediwang mo lenaaneng leno di ikopanye le dikantoro tsa kgaolo fa e le gore le tsona di batla go akarediwa
3: basimane ba rata go tshameka kgwele ya dinaȏ kwa ntle
4: go tshwanetse go nne le lenaneo le le tla rutang bagolo ka dilo tsa go tshwana le tse
5: basimane ba rata go tshameka kgwele ya dinaȏ kwa ntle
6: di diriwa ka letlalo la kgomo kgotsa letlalo la podi kgotsa la nku
7: ke nagana gore motho o tshwanetse go rulaganya go
8: seno se re ama rotlhe mme re tshwanetse go jara maikarabelo mmogo mme rotlhe re thusane go goga le go ntsha naga ya rona mo tobetobeng e e leng mo go yona
9: botshelo jwa diritibatsi ke bo tlisitse mathata mo basimaneng ba bantsi
10: basimane ba tshameka motshameko wa modikologȏ mo lebaleng


## 6. Build vocabulary from your transcripts

In [25]:
def extract_chars(batch):
    all_text = " ".join(batch["transcript"])
    return {"vocab": [list(set(all_text))]}

vocab_set = set()
for split in ["train", "dev"]:
    v = dataset_dict[split].map(
        extract_chars, batched=True, batch_size=-1,
        keep_in_memory=True, remove_columns=dataset_dict[split].column_names,
    )
    for row in v["vocab"]:
        vocab_set.update(row)

vocab_dict = {v: k for k, v in enumerate(sorted(vocab_set))}
vocab_dict["|"] = vocab_dict.pop(" ", len(vocab_dict))
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

print(f"Vocab size: {len(vocab_dict)}")
vocab_dict

Map: 100%|██████████| 500/500 [00:00<00:00, 141193.83 examples/s]

Vocab size: 35


{"'": 1,
 'a': 2,
 'b': 3,
 'c': 4,
 'd': 5,
 'e': 6,
 'f': 7,
 'g': 8,
 'h': 9,
 'i': 10,
 'j': 11,
 'k': 12,
 'l': 13,
 'm': 14,
 'n': 15,
 'o': 16,
 'p': 17,
 'q': 18,
 'r': 19,
 's': 20,
 't': 21,
 'u': 22,
 'v': 23,
 'w': 24,
 'x': 25,
 'y': 26,
 'z': 27,
 'ê': 28,
 'ô': 29,
 'š': 30,
 'ȇ': 31,
 'ȏ': 32,
 '|': 0,
 '[UNK]': 33,
 '[PAD]': 34}

## 7. Build processor

In [26]:
tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|"
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0,
    do_normalize=True, return_attention_mask=True,
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

## 8. Preprocess audio + labels

Same as before, using `processor.tokenizer(...)` directly (no deprecated `as_target_processor()`).

In [27]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    return batch

In [28]:
test_sentence = dataset_dict["train"][0]["transcript"]
test_sentence

'o neetswe tumalanô ya go tsaya loetô la go ya kwa china'

In [29]:
for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].filter(is_valid_transcript, num_proc=2)
    dataset_dict[split] = dataset_dict[split].map(
        prepare_dataset,
        remove_columns=dataset_dict[split].column_names,
        num_proc=2,  # bump to e.g. 4 if you have CPU cores to spare
    )

In [30]:
# for i in range(len(dataset_dict["train"])):
#     ex = dataset_dict["train"][i]

#     assert "input_values" in ex
#     assert "labels" in ex

#     assert len(ex["input_values"]) > 0
#     assert len(ex["labels"]) > 0

In [31]:
dataset_dict["train"].column_names

['input_values', 'input_length', 'labels']

In [32]:
encoded = processor(text=test_sentence).input_ids
decoded = processor.decode(encoded)

print(f"Original: {test_sentence}")
print(f"Decoded:  {decoded}")

Original: o neetswe tumalanô ya go tsaya loetô la go ya kwa china
Decoded:  o netswe tumalanô ya go tsaya loetô la go ya kwa china


In [33]:
print("Vocab size:", len(processor.tokenizer))
print("Pad token:", processor.tokenizer.pad_token)
print("Pad ID:", processor.tokenizer.pad_token_id)

Vocab size: 37
Pad token: [PAD]
Pad ID: 34


In [34]:
print(processor.tokenizer.special_tokens_map)
print(processor.tokenizer.all_special_tokens)
print(processor.tokenizer.all_special_ids)

{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '[UNK]', 'pad_token': '[PAD]', 'word_delimiter_token': '|'}
['<s>', '</s>', '[UNK]', '[PAD]', '|']
[35, 36, 33, 34, 0]


## 9. CTC length-feasibility filter — now ON

This is the single most important fix here. Wav2Vec2-large-XLSR-53
downsamples audio by roughly 320x (5 conv layers with strides
multiplying to ~320). If a clip's frame count after downsampling is
close to or below its label length, CTC has no valid alignment to find —
the loss on that sample balloons and gradients get dragged toward a
degenerate shortcut. Filtering these out before training removes that
source of collapse pressure entirely.

In [35]:
def length_ok(batch):
    # rough CTC feasibility check: downsampled frames must exceed label length
    approx_frames = batch["input_length"] // 320
    return approx_frames > len(batch["labels"])

before_counts = {split: len(dataset_dict[split]) for split in ["train", "dev"]}

for split in ["train", "dev"]:
    dataset_dict[split] = dataset_dict[split].filter(length_ok, num_proc=4)

for split in ["train", "dev"]:
    print(f"{split}: {before_counts[split]} -> {len(dataset_dict[split])} after length filter")

train: 4497 -> 4497 after length filter
dev: 500 -> 500 after length filter


In [36]:
sample = dataset_dict["train"][0]
print(len(sample["input_values"]))
print(len(sample["labels"]))

98749
55


## 10. Data collator

In [37]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

## 11. Metrics (WER / CER)

In [38]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    return {
        "wer": wer_metric.compute(predictions=pred_str, references=label_str),
        "cer": cer_metric.compute(predictions=pred_str, references=label_str),
        "examples": {"prediction": pred_str[:3], "label": label_str[:3]}
    }

## 12. Load model — phased freezing

Instead of unfreezing the whole 24-layer encoder, only the top
`NUM_TRAINABLE_ENCODER_LAYERS` layers + `lm_head` are trainable. This
directly targets the collapse: far fewer free parameters means far less
room for the model to find a cheap degenerate shortcut before it's had
enough steps to learn real acoustic-to-character alignment. Widen this
once you see stable, non-degenerate predictions and have scaled up the
dataset size.

In [39]:
def count_parameters(model):
    trainable_params = 0
    frozen_params = 0
    
    for name, param in model.named_parameters():
        num_params = param.numel()
        if param.requires_grad:
            trainable_params += num_params
        else:
            frozen_params += num_params
            
    total_params = trainable_params + frozen_params
    
    print(f"Total Parameters:     {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f"Frozen Parameters:    {frozen_params:,} ({100 * frozen_params / total_params:.2f}%)")
    
    return trainable_params, frozen_params

# Usage after initializing your model:
# count_parameters(model)

In [40]:
def freeze_transformer_layers(model, freeze_layers=18):
    """
    Freeze the first `freeze_layers` transformer encoder layers.

    Args:
        model: Wav2Vec2ForCTC model
        freeze_layers: Number of encoder layers to freeze.
                       XLS-R 300M has 24 encoder layers.
    """

    encoder_layers = model.wav2vec2.encoder.layers

    # Freeze lower transformer layers
    for layer in encoder_layers[:freeze_layers]:
        for param in layer.parameters():
            param.requires_grad = False

    # Unfreeze upper transformer layers
    for layer in encoder_layers[freeze_layers:]:
        for param in layer.parameters():
            param.requires_grad = True

    # Keep LM head trainable
    for param in model.lm_head.parameters():
        param.requires_grad = True

    # Statistics
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_params = total_params - trainable_params

    total_layers = len(encoder_layers)
    trainable_layers = total_layers - freeze_layers

    print("=" * 60)
    print(f"Transformer layers : {total_layers}")
    print(f"Frozen layers      : {freeze_layers}")
    print(f"Trainable layers   : {trainable_layers}")
    print("-" * 60)
    print(f"Total parameters      : {total_params:,}")
    print(f"Trainable parameters  : {trainable_params:,} "
          f"({100*trainable_params/total_params:.2f}%)")
    print(f"Frozen parameters     : {frozen_params:,} "
          f"({100*frozen_params/total_params:.2f}%)")
    print("=" * 60)

In [41]:
model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    attn_implementation="sdpa",
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.1,
    mask_time_prob=0.1,
    layerdrop=0.1,
)
model.freeze_feature_encoder()  # keep pretrained low-level audio features frozen

# # Freeze all encoder transformer layers, then selectively unfreeze the top N.
# total_layers = len(model.wav2vec2.encoder.layers)
# # for i, layer in enumerate(model.wav2vec2.encoder.layers):
# #     requires_grad = i >= (total_layers - NUM_TRAINABLE_ENCODER_LAYERS)
# #     for param in layer.parameters():
# #         param.requires_grad = requires_grad

# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total = sum(p.numel() for p in model.parameters())
# print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
# print(f"Trainable encoder layers: top {NUM_TRAINABLE_ENCODER_LAYERS} of {total_layers}")

model = model.to("cuda" if torch.cuda.is_available() else "cpu")

Loading weights: 100%|██████████| 422/422 [00:00<00:00, 37364.55it/s]
[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [42]:
#freeze_transformer_layers(model, freeze_layers=18)

In [43]:
count_parameters(model)

Total Parameters:     315,476,645
Trainable Parameters: 311,266,469 (98.67%)
Frozen Parameters:    4,210,176 (1.33%)


(311266469, 4210176)

In [44]:
print("ctc_zero_infinity:", model.config.ctc_zero_infinity)
print("ctc_loss_reduction:", model.config.ctc_loss_reduction)

ctc_zero_infinity: True
ctc_loss_reduction: mean


## 13. Training arguments

- `bf16=True` (was `fp16`): more stable on Ampere, no loss-scaling needed.
- `warmup_ratio` raised and computed against realistic step counts — with
  phased freezing you have far fewer trainable params, so it's worth giving
  the optimizer a real ramp rather than ~4 steps.
- `group_by_length=True`: batches similar-length clips together, cutting
  wasted compute on padding — this is usually the single biggest local
  speed win for variable-length audio.
- `optim="adamw_torch_fused"`: fused CUDA AdamW kernel, meaningfully
  faster per step than the default eager implementation.
- `dataloader_num_workers`: overlaps data loading with GPU compute instead
  of blocking on it every step.

In [45]:
import math
import torch
from transformers import TrainingArguments

def create_training_args(
    output_dir,
    train_dataset,
    num_epochs,
    per_device_train_batch_size,
    per_device_eval_batch_size,
    gradient_accumulation_steps=1,
    learning_rate=3e-5,
    evals_per_epoch=1,
    logs_per_epoch=4,
    saves_per_epoch=1,
    warmup_ratio=0.10,
):
    """
    Create TrainingArguments with step-based intervals computed from epochs.

    Parameters
    ----------
    evals_per_epoch : int
        Number of evaluations per epoch.
    logs_per_epoch : int
        Number of logging events per epoch.
    saves_per_epoch : int
        Number of checkpoints per epoch.
    warmup_ratio : float
        Fraction of total optimizer steps used for warmup.
    """

    world_size = max(torch.cuda.device_count(), 1)

    steps_per_epoch = math.ceil(
        len(train_dataset)
        / (
            per_device_train_batch_size
            * gradient_accumulation_steps
            * world_size
        )
    )

    total_steps = steps_per_epoch * num_epochs

    warmup_steps = max(1, int(total_steps * warmup_ratio))
    eval_steps = max(1, steps_per_epoch // evals_per_epoch)
    logging_steps = max(1, steps_per_epoch // logs_per_epoch)
    save_steps = max(1, steps_per_epoch // saves_per_epoch)
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16 = torch.cuda.is_available() and not torch.cuda.is_bf16_supported()

    print(f"Steps/epoch : {steps_per_epoch}")
    print(f"Total steps : {total_steps}")
    print(f"Warmup      : {warmup_steps}")
    print(f"Eval steps  : {eval_steps}")
    print(f"Log steps   : {logging_steps}")
    print(f"Save steps  : {save_steps}")
    print(f"bf16        : {bf16}")
    print(f"fp16        : {fp16}")

    return TrainingArguments(
        output_dir=output_dir,
        save_strategy="epoch",   # disables checkpoint saving entirely, turn on for prod
        load_best_model_at_end=True,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=per_device_eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        eval_strategy="epoch",
        eval_steps=eval_steps,
        save_steps=save_steps,
        logging_steps=logging_steps,
        learning_rate=learning_rate,
        warmup_steps=warmup_steps,
        num_train_epochs=num_epochs,
        bf16=bf16,
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True,
        length_column_name="input_length",
        train_sampling_strategy="group_by_length",
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        dataloader_persistent_workers=False,
        metric_for_best_model="wer",
        greater_is_better=False,
        push_to_hub=False,
        report_to=["tensorboard"],
        weight_decay=0.01,

    )

In [46]:
# training_args = TrainingArguments(
#     output_dir=OUTPUT_DIR,
#     save_strategy="no",   # disables checkpoint saving entirely, turn on for prod
#     load_best_model_at_end=False,  # disables loading best model at end, turn on for prod
#     per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
#     per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
#     gradient_accumulation_steps=GRAD_ACCUM_STEPS,
#     eval_strategy="steps",
#     eval_steps=10,
#     save_steps=600,
#     logging_steps=10,
#     learning_rate=5e-5,
#     warmup_steps=10
#     num_train_epochs=3,
#     bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
#     fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
#     max_grad_norm=1.0,
#     gradient_checkpointing=True,  # trade speed for VRAM headroom
#     optim="adamw_torch_fused",
#     dataloader_num_workers=4,
#     save_total_limit=2,
#     metric_for_best_model="wer",
#     greater_is_better=False,
#     push_to_hub=False,
#     report_to=[],
# )
# RTX 3060 12GB: batch 4-8 with grad accumulation is a safe start.
# If you hit CUDA OOM, drop per_device_train_batch_size to 2-4.
PER_DEVICE_TRAIN_BATCH = 8
GRAD_ACCUM_STEPS = 1
PER_DEVICE_EVAL_BATCH = 4

training_args = create_training_args(
    output_dir=OUTPUT_DIR,
    train_dataset=dataset_dict["train"],
    num_epochs=40,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
)

Steps/epoch : 563
Total steps : 22520
Warmup      : 2252
Eval steps  : 563
Log steps   : 140
Save steps  : 563
bf16        : True
fp16        : False


In [47]:
from transformers import EarlyStoppingCallback

class DelayedEarlyStoppingCallback(EarlyStoppingCallback):
    def __init__(self, early_stopping_patience=5, start_epoch=5, **kwargs):
        super().__init__(early_stopping_patience=early_stopping_patience, **kwargs)
        self.start_epoch = start_epoch

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        # Ignore evaluation checks until the specified epoch is reached
        if state.epoch is not None and state.epoch < self.start_epoch:
            return
        
        # Proceed with standard early stopping logic after start_epoch
        super().on_evaluate(args, state, control, metrics, **kwargs)

In [48]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    processing_class=processor.feature_extractor,
    callbacks=[DelayedEarlyStoppingCallback(early_stopping_patience=5, start_epoch=5)]
)

## 14. Train

Watch the `examples` field in eval output for the first 2-3 evals — if
predictions are still degenerating into a single repeated character, stop
and check (in order): whether the length filter above actually removed
samples (if it removed ~0, the collapse wasn't a length-feasibility issue
and the next lever is dropping `learning_rate` further or reducing
`NUM_TRAINABLE_ENCODER_LAYERS`); whether `bf16` is actually active (printed
in section 2); and whether the effective batch size
(`PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM_STEPS`) is large enough relative to
dataset size — very small effective batches on a tiny dataset make early
training noisy in a way that can also nudge toward collapse.

In [49]:
trainer.train()

Epoch,Training Loss,Validation Loss,Wer,Cer,Examples
1,5.160498,3.865859,1.000000,1.000000,"{'prediction': ['', '', ''], 'label': ['o gorogile kwa tleliniking phakela o kgona go kry a thuso ka bonako mme o boele gae go sa bonagala gonne ditleliniki tsa mo magaeng di tlala thata', 'dibese mo toropong ya rona di botlhokwa thata ke rata gore tlhwatlhwa ya ditekesi tsa tsone di kwa tlase ga ke rate gore di fitlha lata ebile dingwe tsa tsone ga di mo maemong a itekanetseng', 'ka nako e nngwe ka maswabi karata eo e e botlhajana ga eo kgotsa system e ga e bereke ga o kgone go ifitlhelela gope gape ko ditheong tse dingwe']}"
2,2.974800,2.931031,1.000000,1.000000,"{'prediction': ['', '', ''], 'label': ['o gorogile kwa tleliniking phakela o kgona go kry a thuso ka bonako mme o boele gae go sa bonagala gonne ditleliniki tsa mo magaeng di tlala thata', 'ano ke maikarabelo a rona rotlhe re le puso dikolo ditheo tsa thuto e kgolwane batsadi malapa le masika baopedi badiragatsi le botlhe mo setšhabeng', 'ka le lengwe la matsatsi ke ne ka leba lebenkeleng go ya go reka mae mme ke ne ka fitlhela tlhotlhwa ya ona e le kwa godimo thata']}"
3,2.886841,2.885130,1.000000,1.000000,"{'prediction': ['', '', ''], 'label': ['o gorogile kwa tleliniking phakela o kgona go kry a thuso ka bonako mme o boele gae go sa bonagala gonne ditleliniki tsa mo magaeng di tlala thata', 'ano ke maikarabelo a rona rotlhe re le puso dikolo ditheo tsa thuto e kgolwane batsadi malapa le masika baopedi badiragatsi le botlhe mo setšhabeng', 'ka le lengwe la matsatsi ke ne ka leba lebenkeleng go ya go reka mae mme ke ne ka fitlhela tlhotlhwa ya ona e le kwa godimo thata']}"
4,2.790079,2.619068,0.999562,0.933096,"{'prediction': ['aaaaaa', 'aaaaaaaaaaa', 'aa'], 'label': ['o gorogile kwa tleliniking phakela o kgona go kry a thuso ka bonako mme o boele gae go sa bonagala gonne ditleliniki tsa mo magaeng di tlala thata', 'phuthego e ne e beilwe go keteka nako le matsapa a setlhopha se a dirisitseng go tsweletse pele lenaneo la go bona balemirui ba ba ka tlhabololwang', 'mo kgweding eno go letsatsi le le bee tsweng thoko go keteka go tsalwa ga kgololosego le temokerasi le diphitlhelelo tsa maaforika']}"
5,0.995123,0.689508,0.492775,0.150437,"{'prediction': ['fa o gorogile kwa tlenekeng phakela o kgona go o keria thuso ka monako mme o bwelegae e go sa bonagala go ne ditliiniki tsa magae di tlala thata', 'a no ke maikarebelo a rona rotlhegr lepuso dikolo di theo tsa thuto e kgolwaneg ba tsadi malapa le maseka baopedi ba diragats le bootlhe mo sethabenng', 'mo setsung tsa gaeso tshakogaye reaparedi a pago tsa setlhopa sha seng dwebelemme a a diitsi difetoge le kank le katsa tsila ka jeno'], 'label': ['o gorogile kwa tleliniking phakela o kgona go kry a thuso ka bonako mme o boele gae go sa bonagala gonne ditleliniki tsa mo magaeng di tlala thata', 'ano ke maikarabelo a rona rotlhe re le puso dikolo ditheo tsa thuto e kgolwane batsadi malapa le masika baopedi badiragatsi le botlhe mo setšhabeng', 'mo setsong sa gaetsho tsa ko gae re apara diaparo tsa setlhopha sa sendebele mme a di se di fetoge le ka tsatsi la kajeno']}"
6,0.745462,0.516591,0.404904,0.124617,"{'prediction': ['ao goreogile kwatlenekeng phakela o kgona go okeria thuso ka monako mme o boelegae go sa bonagala go ne ditleineki tsa magaeng ditlala thata', 'ano ke maikarebelo a rona rotlhege lepuso dikolo dithe tsa thuto e kgolwaneg batsadi malapa le maseka baopedi ba diragats le bootlhe mo sethabeng', 'ka lelengwela matsatsi ke ne ka leba lebenkelenggo ya gore ka maimme ke ne ka fitlhela tlhotlhwa ya ona e le kwa godimotata'], 'label': ['o gorogile kwa tleliniking phakela o kgona go kry a thuso ka bonako mme o boele gae go sa bonagala gonne ditleliniki tsa mo magaeng di tlala thata', 'ano ke maikarabelo a rona rotlhe re le puso dikolo ditheo tsa thuto e kgolwane batsadi malapa le masika baopedi badiragatsi le botlhe mo setšhabeng', 'ka le lengwe la matsatsi ke ne ka leba lebenkeleng go ya go reka mae mme ke ne ka fitlhela tl

[transformers] Trainer is attempting to log a value of "{'prediction': ['', '', ''], 'label': ['o gorogile kwa tleliniking phakela o kgona go kry a thuso ka bonako mme o boele gae go sa bonagala gonne ditleliniki tsa mo magaeng di tlala thata', 'dibese mo toropong ya rona di botlhokwa thata ke rata gore tlhwatlhwa ya ditekesi tsa tsone di kwa tlase ga ke rate gore di fitlha lata ebile dingwe tsa tsone ga di mo maemong a itekanetseng', 'ka nako e nngwe ka maswabi karata eo e e botlhajana ga eo kgotsa system e ga e bereke ga o kgone go ifitlhelela gope gape ko ditheong tse dingwe']}" of type <class 'dict'> for key "eval/examples" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.55s/it]
[transformers] Trainer is attempting to log a value of "{'prediction': ['', '', ''], 'label': ['o gorogile kwa tleliniking phakela o kgona go kry a thuso ka bonako mme o boele gae go sa b

TrainOutput(global_step=22520, training_loss=0.7997385237610785, metrics={'train_runtime': 16034.4677, 'train_samples_per_second': 11.218, 'train_steps_per_second': 1.404, 'total_flos': 4.496695003860917e+19, 'train_loss': 0.7997385237610785, 'epoch': 40.0})

In [50]:
trainer.state.log_history

[{'loss': 15.487847028459822,
  'grad_norm': 9.010086059570312,
  'learning_rate': 1.8516873889875667e-06,
  'epoch': 0.24866785079928952,
  'step': 140},
 {'loss': 14.04915248325893,
  'grad_norm': 16.869678497314453,
  'learning_rate': 3.716696269982238e-06,
  'epoch': 0.49733570159857904,
  'step': 280},
 {'loss': 7.868735177176339,
  'grad_norm': 22.453914642333984,
  'learning_rate': 5.58170515097691e-06,
  'epoch': 0.7460035523978685,
  'step': 420},
 {'loss': 5.160497610909598,
  'grad_norm': 6.653079032897949,
  'learning_rate': 7.4467140319715815e-06,
  'epoch': 0.9946714031971581,
  'step': 560},
 {'eval_loss': 3.865858793258667,
  'eval_wer': 1.0,
  'eval_cer': 1.0,
  'eval_examples': {'prediction': ['', '', ''],
   'label': ['o gorogile kwa tleliniking phakela o kgona go kry a thuso ka bonako mme o boele gae go sa bonagala gonne ditleliniki tsa mo magaeng di tlala thata',
    'dibese mo toropong ya rona di botlhokwa thata ke rata gore tlhwatlhwa ya ditekesi tsa tsone di kwa

## 15. Save

In [51]:
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.46s/it]

Saved to ./wav2vec2-300-tsn


In [52]:

processor = Wav2Vec2Processor.from_pretrained(OUTPUT_DIR)
model = Wav2Vec2ForCTC.from_pretrained(OUTPUT_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

Loading weights: 100%|██████████| 424/424 [00:00<00:00, 11738.20it/s]


Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, eleme

## 16. Quick sanity-check inference

In [53]:
import soundfile as sf

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
model.to(device)

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, eleme

In [54]:
sample = dataset_dict["dev"].select(range(1))[0]
input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
print("Raw predicted IDs:", predicted_ids[0].tolist())
print("Unique IDs predicted:", set(predicted_ids[0].tolist()))
print("Pad/blank token ID:", processor.tokenizer.pad_token_id)

Raw predicted IDs: [34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 13, 6, 34, 34, 34, 34, 34, 21, 20, 20, 16, 34, 34, 34, 34, 34, 34, 34, 8, 8, 16, 0, 0, 0, 34, 34, 34, 13, 13, 6, 0, 0, 34, 34, 34, 13, 6, 34, 34, 34, 34, 34, 34, 34, 12, 9, 9, 22, 34, 34, 34, 34, 21, 20, 20, 9, 24, 2, 2, 34, 34, 34, 34, 34, 15, 6, 0, 0, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 13, 13, 6, 0, 0, 34, 34, 34, 12, 9, 9, 22, 34, 34, 34, 19, 22, 34, 34, 34, 34, 34, 14, 6, 6, 34, 34, 34, 34, 34, 21, 20, 20, 2, 2, 0, 0, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 13, 6, 34, 34, 34, 34, 34, 34, 34, 20, 20, 22, 34, 34, 34, 34, 34, 7, 22, 22, 34, 34, 34, 34, 21, 21, 20, 34, 16, 34, 34, 34, 34, 34, 34, 34, 8, 16, 0, 0, 34, 34, 34, 34, 13, 34, 2, 0, 0, 0, 34, 34, 34, 34, 34, 8, 2, 2, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 8, 16, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34

# Test 1: preprocessed dev split

In [55]:
def evaluate_split(split="dev_test", num_samples=5):
    print(f"=== Evaluation on {split} split ===\n")
    test_samples = dataset_dict[split].select(range(num_samples))

    pred_texts = []
    ref_texts = []

    for i, sample in enumerate(test_samples):
        input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(input_values).logits

        predicted_ids = torch.argmax(logits, dim=-1)
        predicted_text = processor.tokenizer.decode(predicted_ids[0])
        actual_text = processor.tokenizer.decode(sample["labels"], group_tokens=False)

        pred_texts.append(predicted_text)
        ref_texts.append(actual_text)

        print(f"--- Sample {i+1} ---")
        print(f"Predicted: {predicted_text}")
        print(f"Actual:    {actual_text}\n")

    wer = wer_metric.compute(predictions=pred_texts, references=ref_texts)
    cer = cer_metric.compute(predictions=pred_texts, references=ref_texts)

    print(f"WER: {wer:.4f}")
    print(f"CER: {cer:.4f}")

    return {"wer": wer, "cer": cer, "predictions": pred_texts, "references": ref_texts}

In [56]:
evaluate_split(split="dev", num_samples=100)

=== Evaluation on dev split ===

--- Sample 1 ---
Predicted: letsogo le lekhutshwane le khurumetsa lesufutsogo la gago
Actual:    letsogo le lekhutshwane le khurumetsa lesufutsogo la gago

--- Sample 2 ---
Predicted: lepodisa le dirisitse mosifa wa gagwe go phatlalatsa batšha mo ntweng
Actual:    lepodisa le dirisitse mosifa wa gagwe go phatlalatsa batšha mo ntweng

--- Sample 3 ---
Predicted: gompieno kêrêkê e ne e tletse tota
Actual:    gompieno kêrêkê e ne e tletse tota

--- Sample 4 ---
Predicted: banna botlhê ba simolotse patlô ya mosetsana yo o timetseng
Actual:    banna botlhê ba simolotse patlô ya mosetsana yo o timetseng

--- Sample 5 ---
Predicted: bagolo ba ne ba gakgamala fa pusô e nfedisa thutô ya bodumedi kwa dikolong
Actual:    bagolo ba ne ba gakgamala fa pusô e fedisa thutô ya bodumedi kwa dikolong

--- Sample 6 ---
Predicted: moȇng wa dikgwȇbȏ pȏtlana o gorogile maabane kwa kopanong ya dikgwȇbȏ
Actual:    moȇng wa dikgwȇbȏ pȏtlana o gorogile maabane kwa kopanong ya di

{'wer': 0.12654545454545454,
 'cer': 0.04491595480848719,
 'predictions': ['letsogo le lekhutshwane le khurumetsa lesufutsogo la gago',
  'lepodisa le dirisitse mosifa wa gagwe go phatlalatsa batšha mo ntweng',
  'gompieno kêrêkê e ne e tletse tota',
  'banna botlhê ba simolotse patlô ya mosetsana yo o timetseng',
  'bagolo ba ne ba gakgamala fa pusô e nfedisa thutô ya bodumedi kwa dikolong',
  'moȇng wa dikgwȇbȏ pȏtlana o gorogile maabane kwa kopanong ya dikgwȇbȏ',
  'baithuti ba ithuta ka molawana wa motheo mo dipalong',
  'jaaka dikolo ditsweletsi ba ne ba tshwanelwa ke go jara joko e e se nkana ka sepo ya go ithutela bana ba bona letsatsi le ngwe le lengwe',
  'motlhatlhaledi wa baeloji o neela baithuti tirwana ya mosola wa penselene',
  'ba dirisitse sepalangwasa maoto go rwala dithôtô',
  'loitumȇlo le kagiso ke setlhôpô kwa mo botselong',
  'tebogȏ o dumeletse setlhopha sa gagwe sa rakabi go dirisa le galê go tshameka',
  'ka lako lwecstvin bantobko o ne a bolawa a tswaletswe kw

# Test 2: unprocessessed dev split

In [57]:
print_duration_buckets(dataset_dict, splits=["dev_test"], bucket_size=1)

dev_test
total duration: 92776.385634 seconds
total duration: 1546.2730939 minutes
total duration: 25.77121823166667 hours
total samples: 5264

    2.0 -   3.0 sec:     12 samples (  0.2%)
    3.0 -   4.0 sec:    113 samples (  2.1%)
    4.0 -   5.0 sec:    427 samples (  8.1%)
    5.0 -   6.0 sec:    572 samples ( 10.9%)
    6.0 -   7.0 sec:    508 samples (  9.7%)
    7.0 -   8.0 sec:    357 samples (  6.8%)
    8.0 -   9.0 sec:    229 samples (  4.4%)
    9.0 -  10.0 sec:    160 samples (  3.0%)
   10.0 -  11.0 sec:    183 samples (  3.5%)
   11.0 -  12.0 sec:    137 samples (  2.6%)
   12.0 -  13.0 sec:    125 samples (  2.4%)
   13.0 -  14.0 sec:    134 samples (  2.5%)
   14.0 -  15.0 sec:    119 samples (  2.3%)
   15.0 -  16.0 sec:    102 samples (  1.9%)
   16.0 -  17.0 sec:    102 samples (  1.9%)
   17.0 -  18.0 sec:     96 samples (  1.8%)
   18.0 -  19.0 sec:    108 samples (  2.1%)
   19.0 -  20.0 sec:     98 samples (  1.9%)
   20.0 -  21.0 sec:     91 samples (  1.7%)
 

In [58]:
dataset_dict = filter_by_duration(dataset_dict, min_seconds=4, max_seconds=15, splits=["dev_test"], num_proc=2)
dataset_dict = sample_n(dataset_dict, {"dev_test": 100})

In [59]:
print_duration_buckets(dataset_dict, splits=["dev_test"], bucket_size=1)

dev_test
total duration: 752.157804 seconds
total duration: 12.535963400000002 minutes
total duration: 0.20893272333333335 hours
total samples: 100

    4.0 -   5.0 sec:     21 samples ( 21.0%)
    5.0 -   6.0 sec:     16 samples ( 16.0%)
    6.0 -   7.0 sec:     17 samples ( 17.0%)
    7.0 -   8.0 sec:     14 samples ( 14.0%)
    8.0 -   9.0 sec:      5 samples (  5.0%)
    9.0 -  10.0 sec:      7 samples (  7.0%)
   10.0 -  11.0 sec:      3 samples (  3.0%)
   11.0 -  12.0 sec:      7 samples (  7.0%)
   12.0 -  13.0 sec:      3 samples (  3.0%)
   13.0 -  14.0 sec:      3 samples (  3.0%)
   14.0 -  15.0 sec:      4 samples (  4.0%)



In [60]:
import torch

def evaluate_raw_split(split="dev_test", num_samples=5, print_n=5, normalize_fn=None):
    """
    Args:
        split (str): dataset split to evaluate on.
        num_samples (int): number of samples to evaluate (use len(dataset_dict[split]) for full split).
        print_n (int): number of sample predictions to print, regardless of num_samples.
        normalize_fn (callable, optional): function to apply to reference transcripts
            before WER/CER computation, to match training-time normalization.
    """
    print(f"=== Evaluation on {split} split ===\n")

    samples = dataset_dict[split].select(range(num_samples))

    pred_texts = []
    ref_texts = []

    model.eval()

    for i, sample in enumerate(samples):
        # Raw audio
        audio = sample["audio"]["array"]

        # Convert to model inputs
        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True,
        )

        input_values = inputs.input_values.to(device)

        with torch.no_grad():
            logits = model(input_values).logits

        predicted_ids = torch.argmax(logits, dim=-1)

        # Decode prediction
        predicted_text = processor.batch_decode(predicted_ids)[0]

        # Reference transcription
        actual_text = sample["transcript"]  # <-- change if your transcript column has another name

        if normalize_fn is not None:
            actual_text = normalize_fn(actual_text)

        pred_texts.append(predicted_text)
        ref_texts.append(actual_text)

        if i < print_n:
            print(f"--- Sample {i+1} ---")
            print(f"Predicted: {predicted_text}")
            print(f"Actual:    {actual_text}\n")

    if num_samples > print_n:
        print(f"... ({num_samples - print_n} more samples evaluated but not printed)\n")

    wer = wer_metric.compute(
        predictions=pred_texts,
        references=ref_texts,
    )

    cer = cer_metric.compute(
        predictions=pred_texts,
        references=ref_texts,
    )

    print(f"WER: {wer:.4f}")
    print(f"CER: {cer:.4f}")

    return {
        "wer": wer,
        "cer": cer,
        "predictions": pred_texts,
        "references": ref_texts,
    }

In [61]:
evaluate_raw_split(split="dev_test", num_samples=100, print_n=5)

=== Evaluation on dev_test split ===

--- Sample 1 ---
Predicted: go namile maoto mo metseng ka fa gare ga lelapa le ditsala tsa gagwe
Actual:    mme o namile maoto mo mmetseng ka fa gare ga lelapa le ditsala tsa gagwe

--- Sample 2 ---
Predicted: basimane ba tshameka motshameko wa modikologȏ mo lebaleng
Actual:    basimane ba tshameka motshameko wa modikologȏ mo lebaleng

--- Sample 3 ---
Predicted: seno ke ka ditebogo go tswa mo katisong le thusong eo a e neilweng ke ba lenaane la tshegetso le tlhabololo ya
Actual:    seno ke ka ditebogo go tswa mo katisong le thu song eo a e neilweng ke ba lenaane la tshegetso le tlhabololo ya

--- Sample 4 ---
Predicted: morwa wa gagwe o nopile kgobokanô ya matlapa a le tlhano a simolola go konopa
Actual:    morwa wa gagwe o nopile kgobokanô ya matlapa a le tlhano a simolola go konopa

--- Sample 5 ---
Predicted: bagolo ba ne ba gakgamala fa pusô e fedisa thutô ya bodumedi kwa dikolong
Actual:    bagolo ba ne ba gakgamala fa pusô e fedisa thutô ya 

{'wer': 0.158852344296711,
 'cer': 0.0684323742941651,
 'predictions': ['go namile maoto mo metseng ka fa gare ga lelapa le ditsala tsa gagwe',
  'basimane ba tshameka motshameko wa modikologȏ mo lebaleng',
  'seno ke ka ditebogo go tswa mo katisong le thusong eo a e neilweng ke ba lenaane la tshegetso le tlhabololo ya',
  'morwa wa gagwe o nopile kgobokanô ya matlapa a le tlhano a simolola go konopa',
  'bagolo ba ne ba gakgamala fa pusô e fedisa thutô ya bodumedi kwa dikolong',
  'maemo fa a le jaana a tlhoka gore re se ikaketse a tlhoka gore re dirisane mmogo go nna le go fetlha dikgogakgogama',
  'mmagwe kgapô morago ga tlhalȏ',
  'mothapi wa lefapha la thutȏ o thapile basadi ba ba raro',
  'ke nna mo metseng magaeng ka jalo go tsamaya ka dinao ke kgona gore go dirisang thata mme bile re itekanetse e le rurika labaka la gore re phelare tsamaya ya re taboga jalo le jalo',
  'kutlwanȏ rokile morokô wa mosese wa gagwe ka tlhale',
  'borra saense ba dumela fela fa ba kgonne go bona poe

# Test 2: raw audio files (informal validation)

In [62]:
import soundfile as sf
import torch
import torchaudio.transforms as T


def transcribe_audio(
    audio_path,
    model,
    processor,
    device,
    sampling_rate=16000,
    return_logits=False,
):
    model.eval()

    # Load using PySoundFile directly
    audio, native_sr = sf.read(audio_path, dtype="float32")

    # Convert stereo to mono if needed
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Resample if native sampling rate differs from target sampling rate
    if native_sr != sampling_rate:
        audio_tensor = torch.from_numpy(audio).float().unsqueeze(0)
        resampler = T.Resample(orig_freq=native_sr, new_freq=sampling_rate)
        audio = resampler(audio_tensor).squeeze(0).numpy()

    # Feature extraction
    inputs = processor(
        audio,
        sampling_rate=sampling_rate,
        return_tensors="pt",
        padding=True,
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]

    print(f"\nFile: {audio_path}")
    print(f"Prediction:\n{transcription}")

    if return_logits:
        return transcription, logits

    return transcription

In [63]:
transcribe_audio(
    "rc4.wav",
    model,
    processor,
    device,
)


File: rc4.wav
Prediction:
tumalang o kae


'tumalang o kae'

In [64]:
transcribe_audio(
    "rc5.wav",
    model,
    processor,
    device,
)


File: rc5.wav
Prediction:
ke tshwere ke tlala


'ke tshwere ke tlala'

In [65]:
def transcribe_audio_file(filepath):
    audio, sr = sf.read(filepath)
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt").input_values.to(device)

    with torch.no_grad():
        logits = model(inputs).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    return processor.tokenizer.decode(predicted_ids[0])

In [66]:
def test_on_raw_files(filepaths):
    print("=== Transcription on raw audio files ===\n")
    for path in filepaths:
        prediction = transcribe_audio_file(path)
        print(f"File: {path}")
        print(f"Predicted: {prediction}\n")

In [67]:
raw_files = [
    "/home/khotso/data/validation_clips/clip1.wav",
    "/home/khotso/data/validation_clips/clip2.wav",
]
# test_on_raw_files(raw_files)